# Dashboard Data Layer Validation

This notebook validates the dashboard-ready datasets produced from PostgreSQL workforce data and retention-model outputs.

The dashboard data layer includes:

- Workforce overview KPIs
- Department headcount
- Location headcount
- Recruiting funnel metrics
- Requisition metrics
- Retention risk summaries
- Employee-level retention risk scores
- Model performance metrics

The employee risk scores are intended for dashboard prioritization. Model performance continues to be evaluated using the held-out test set rather than the full scored population.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DASHBOARD_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dashboard"
)


overview = pd.read_csv(
    DASHBOARD_DIR
    / "overview_kpis.csv"
)


department = pd.read_csv(
    DASHBOARD_DIR
    / "headcount_by_department.csv"
)


location = pd.read_csv(
    DASHBOARD_DIR
    / "headcount_by_location.csv"
)


recruiting = pd.read_csv(
    DASHBOARD_DIR
    / "recruiting_funnel.csv"
)


requisitions = pd.read_csv(
    DASHBOARD_DIR
    / "requisition_metrics.csv"
)


risk_summary = pd.read_csv(
    DASHBOARD_DIR
    / "retention_risk_summary.csv"
)


risk_employees = pd.read_csv(
    DASHBOARD_DIR
    / "retention_risk_employees.csv"
)


model_performance = pd.read_csv(
    DASHBOARD_DIR
    / "model_performance.csv"
)


model_summary = pd.read_csv(
    DASHBOARD_DIR
    / "model_summary.csv"
)


print(
    "Dashboard files loaded successfully."
)

Dashboard files loaded successfully.


## 1. Workforce overview

In [2]:
overview

,as_of_date,active_headcount,active_department_count,active_location_count,terminations_2025,turnover_rate_2025_percent,open_requisitions,total_applications,hired_applications,retention_snapshot_population,high_risk_employees,medium_risk_employees,low_risk_employees
0,2026-06-30,8287,8,5,570,7.73,120,52436,10000,7386,739,1477,5170


## 2. Headcount validation

In [3]:
active_headcount = int(
    overview.loc[
        0,
        "active_headcount",
    ]
)


department_total = int(
    department[
        "active_headcount"
    ]
    .sum()
)


location_total = int(
    location[
        "active_headcount"
    ]
    .sum()
)


print(
    "Overview active headcount:",
    active_headcount,
)


print(
    "Department total:",
    department_total,
)


print(
    "Location total:",
    location_total,
)

Overview active headcount: 8287
Department total: 8287
Location total: 8287


## 3. Recruiting funnel

In [4]:
recruiting

,application_source,total_applications,hired,rejected,withdrawn,offer_declined,in_process,position_cancelled,hire_rate_percent
0,LinkedIn,14715,2416,5410,1655,996,2139,2099,16.42
1,Company Careers Page,11006,1966,3970,1205,726,1586,1553,17.86
2,Indeed,8933,1237,3280,1092,637,1335,1352,13.85
3,Employee Referral,6464,2269,1788,569,349,725,764,35.10
4,University Recruiting,3747,811,1284,413,247,500,492,21.64
5,Staffing Agency,3503,522,1236,438,267,552,488,14.90
6,Professional Association,2182,508,754,208,135,300,277,23.28
7,Job Fair,1886,271,694,222,130,273,296,14.37


## 4. Requisition metrics

In [5]:
requisitions

,requisition_status,requisition_count,target_headcount,average_days_open
0,Filled,2409,10000,93.0
1,Cancelled,250,921,50.0
2,Open,120,542,61.5


## 5. Retention risk segments

In [6]:
risk_summary

,risk_segment,employee_count,average_attrition_probability,average_tenure_years,average_base_salary,population_percent
0,High,739,0.735324,0.870785,76423.545332,10.005416
1,Medium,1477,0.625445,1.249546,82413.473257,19.997292
2,Low,5170,0.348124,2.709466,98793.907157,69.997292


In [7]:
risk_employees[
    "risk_segment"
].value_counts()

risk_segment
Low       5170
Medium    1477
High       739
Name: count, dtype: int64

## 6. Highest-ranked retention risks

In [8]:
risk_employees[
    [
        "employee_id",
        "hire_department_name",
        "hire_job_family",
        "tenure_years",
        "attrition_probability",
        "risk_segment",
    ]
].head(20)

,employee_id,hire_department_name,hire_job_family,tenure_years,attrition_probability,risk_segment
0,103686,Manufacturing,Manufacturing,1.02,0.901905,High
1,107952,Human Resources,Analytics,1.19,0.874157,High
2,103130,Manufacturing,Manufacturing,1.13,0.872600,High
3,109873,Customer Support,Customer Support,1.46,0.864995,High
4,102214,Manufacturing,Manufacturing,0.28,0.855480,High
5,108055,Human Resources,Analytics,0.48,0.850528,High
6,103424,Manufacturing,Engineering,0.06,0.848353,High
7,100760,Engineering,Engineering,0.08,0.847551,High
8,101205,Engineering,Engineering,0.25,0.842157,High
9,101264,Engineering,Engineering,0.05,0.840639,High


## 7. Model performance

In [9]:
model_performance

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,true_negative,false_positive,false_negative,true_positive,selected_model
0,Logistic Regression,0.5981,0.1376,0.712,0.2306,0.6968,0.1709,795,558,36,89,True
1,Random Forest,0.8904,0.1754,0.080,0.1099,0.7017,0.1662,1306,47,115,10,False
2,Gradient Boosting,0.6962,0.1524,0.568,0.2403,0.6942,0.1645,958,395,54,71,False


In [10]:
model_summary

,selected_model,selection_metric,recommended_classification_threshold,medium_risk_threshold,high_risk_threshold,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,Logistic Regression,PR-AUC,0.5,0.571982,0.685568,0.5981,0.1376,0.712,0.2306,0.6968,0.1709


## 8. Dashboard data validation

In [11]:
dashboard_checks = pd.Series(
    {
        "overview contains one row": (
            len(
                overview
            )
            == 1
        ),

        "department headcount matches overview": (
            department_total
            == active_headcount
        ),

        "location headcount matches overview": (
            location_total
            == active_headcount
        ),

        "recruiting data is not empty": (
            len(
                recruiting
            )
            > 0
        ),

        "requisition data is not empty": (
            len(
                requisitions
            )
            > 0
        ),

        "three risk segments appear": (
            set(
                risk_employees[
                    "risk_segment"
                ]
            )
            == {
                "Low",
                "Medium",
                "High",
            }
        ),

        "risk probabilities are valid": (
            risk_employees[
                "attrition_probability"
            ]
            .between(
                0,
                1,
            )
            .all()
        ),

        "exactly one model is selected": (
            model_performance[
                "selected_model"
            ]
            .sum()
            == 1
        ),

        "model summary contains one row": (
            len(
                model_summary
            )
            == 1
        ),
    },
    name="passed",
)


dashboard_checks

overview contains one row                True
department headcount matches overview    True
location headcount matches overview      True
recruiting data is not empty             True
requisition data is not empty            True
three risk segments appear               True
risk probabilities are valid             True
exactly one model is selected            True
model summary contains one row           True
Name: passed, dtype: bool

In [12]:
if dashboard_checks.all():

    print(
        "All dashboard data validation "
        "checks passed."
    )

else:

    print(
        "One or more dashboard data "
        "checks failed."
    )

All dashboard data validation checks passed.
